# Knowledge Graph Tutorial 

### Raw data

In [1]:
text = """
Albert Einstein was born in Germany.
Albert Einstein worked at Princeton University.
Marie Curie was born in Poland.
Marie Curie won the Nobel Prize.
"""

### Extract entities (NER)

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

entities = [(ent.text, ent.label_) for ent in doc.ents]

print(entities)

ModuleNotFoundError: No module named 'spacy'

### Simple relation extraction

In [ ]:
triples = []

for sent in doc.sents:
    sent_doc = nlp(sent.text)
    
    persons = [ent.text for ent in sent_doc.ents if ent.label_ == "PERSON"]
    locations = [ent.text for ent in sent_doc.ents if ent.label_ in ["GPE"]]
    
    if "born" in sent.text and persons and locations:
        triples.append((persons[0], "born_in", locations[0]))
        
    if "worked" in sent.text:
        orgs = [ent.text for ent in sent_doc.ents if ent.label_ == "ORG"]
        if persons and orgs:
            triples.append((persons[0], "worked_at", orgs[0]))

print(triples)

### Build the knowledge graph

In [ ]:
import networkx as nx

G = nx.DiGraph()

for h, r, t in triples:
    G.add_edge(h, t, relation=r)

print(G.edges(data=True))

### Save triples for embedding training

In [ ]:
import pandas as pd

df = pd.DataFrame(triples, columns=["head", "relation", "tail"])
df.to_csv("kg_triples.tsv", sep="\t", index=False, header=False)

### Train knowledge graph embeddings

In [ ]:
from pykeen.pipeline import pipeline

result = pipeline(
    training="kg_triples.tsv",
    model="TransE",
    training_kwargs=dict(num_epochs=100),
)

print(result)

### Query the trained model

In [ ]:
model = result.model

# embedding of entity
einstein_embedding = model.entity_representations[0](
    indices=model.triples_factory.entity_to_id["Albert Einstein"]
)

print(einstein_embedding)

### Visualizing the graph

In [ ]:
import matplotlib.pyplot as plt

pos = nx.spring_layout(G)

nx.draw(G, pos, with_labels=True, node_color="lightblue")

edge_labels = nx.get_edge_attributes(G,'relation')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

plt.show()